In [29]:
import pandas as pd
import numpy as np
import os
import time
from backtester import backtest

DATA_DIR = "./data"


In [ ]:
class Model:
    def __init__(self, ops):
        self.bars_seen_train         = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_train.parquet"), engine="fastparquet")
        self.bars_unseen_train       = pd.read_parquet(os.path.join(DATA_DIR, "bars_unseen_train.parquet"), engine="fastparquet")
        self.bars_seen_public_test   = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_public_test.parquet"), engine="fastparquet")
        self.bars_seen_private_test  = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_private_test.parquet"), engine="fastparquet")

        self.headlines_seen_train        = pd.read_parquet(os.path.join(DATA_DIR, "headlines_seen_train.parquet"), engine="fastparquet")
        self.headlines_unseen_train      = pd.read_parquet(os.path.join(DATA_DIR, "headlines_unseen_train.parquet"), engine="fastparquet")
        self.headlines_seen_public_test  = pd.read_parquet(os.path.join(DATA_DIR, "headlines_seen_public_test.parquet"), engine="fastparquet")
        self.headlines_seen_private_test = pd.read_parquet(os.path.join(DATA_DIR, "headlines_seen_private_test.parquet"), engine="fastparquet")

        self.sentiment_public_test = pd.read_csv("./anna/sentiment_overview.csv")
        self.sentiment_private_test = pd.read_csv("./anna/sentiment_overview.csv")
        self.ops = ops
        print("bars_seen_train:",         self.bars_seen_train.shape)
        print("bars_unseen_train:",       self.bars_unseen_train.shape)
        print("bars_seen_public_test:",   self.bars_seen_public_test.shape)
        print("bars_seen_private_test:",  self.bars_seen_private_test.shape)
        print("headlines_seen_train:",        self.headlines_seen_train.shape)
        print("headlines_unseen_train:",      self.headlines_unseen_train.shape)
        print("headlines_seen_public_test:",  self.headlines_seen_public_test.shape)
        print("headlines_seen_private_test:", self.headlines_seen_private_test.shape)


    def fit(self):
        lags = self.ops["lag"]
        
        # Check autocorrelation and distribution of the log returns 
        def autocorrelation(df, lags = 1):
            autocorrs = []
            for col in df.columns:
                series = df[col].dropna()
                if len(series) > lags:
                    autocorrs.append(series.autocorr(lag=lags))
            return autocorrs
        
        open_price_df = self.bars_seen_train[["session", "bar_ix", "open"]]
        open_price_pivot = open_price_df.pivot(index="bar_ix", columns="session", values="open")


        log_ret_open = np.log(open_price_pivot / open_price_pivot.shift(1)).dropna()

        autocorr_lag1 = autocorrelation(df=log_ret_open, lags=lags)

        # print 5 most and least autocorrelated sessions
        autocorr_df = pd.DataFrame({
            "session": log_ret_open.columns,
            "autocorr_lag1": autocorr_lag1
        })  
        autocorr_df = autocorr_df.dropna().sort_values("autocorr_lag1", ascending=False).reset_index(drop=True)
        print("Top 5 most autocorrelated sessions (lag 1):")
        print(autocorr_df.head(5))
        print("\nTop 5 least autocorrelated sessions (lag 1):")
        print(autocorr_df.tail(5))

    def identify_firm(self, bars_df, sentiment_df, session_of_interest=2):
        price_df = bars_df[bars_df["session"] == session_of_interest].copy().copy()
        news_df = sentiment_df[sentiment_df["session"] == session_of_interest].copy()

        # ---------------------------------------------------------
        # STEP 2: Convert sentiment probabilities into one score
        # positive = +1
        # negative = -1
        # neutral = 0
        # ---------------------------------------------------------
        news_df["sentiment_score"] = (
            news_df["prob_positive"] - news_df["prob_negative"]
        )

        # ---------------------------------------------------------
        # STEP 3: Aggregate sentiment by bar_ix and company
        # ---------------------------------------------------------
        sentiment = (
            news_df.groupby(["headline", "bar_ix"], as_index=False)
            .agg(sentiment_score=("sentiment_score", "mean"))
        )

        # ---------------------------------------------------------
        # STEP 4: Daily close price
        # (average close across sessions inside same bar_ix)
        # ---------------------------------------------------------
        daily_px = (
            price_df.groupby("bar_ix", as_index=False)
            .agg(close=("close", "mean"))
            .sort_values("bar_ix")
        )

        # ---------------------------------------------------------
        # STEP 5: FUTURE 5-DAY RETURN
        # price change from today to +5 bars
        # ---------------------------------------------------------
        daily_px["future_close_5d"] = daily_px["close"].shift(-3)

        daily_px["future_ret_5d"] = (
            daily_px["future_close_5d"] - daily_px["close"]
        ) / daily_px["close"]

        price_move = daily_px[["bar_ix", "future_ret_5d"]].dropna()

        # ---------------------------------------------------------
        # STEP 6: Merge each company sentiment with unknown price
        # ---------------------------------------------------------
        merged = sentiment.merge(price_move, on="bar_ix", how="inner")

        # ---------------------------------------------------------
        # STEP 7: Score companies
        # ---------------------------------------------------------
        results = []

        for company, g in merged.groupby("headline"):

            if len(g) < 2:
                continue

            corr = g["sentiment_score"].corr(g["future_ret_5d"])

            sent_sign = np.sign(g["sentiment_score"])
            px_sign   = np.sign(g["future_ret_5d"])

            hitrate = (sent_sign == px_sign).mean()

            # stronger weight on correlation
            score = 0.75 * corr + 0.25 * hitrate

            results.append({
                "company": company,
                "n_obs": len(g),
                "corr_5d": corr,
                "hitrate_5d": hitrate,
                "score": score
            })

        results_df = (
            pd.DataFrame(results)
            .sort_values("score", ascending=False)
            .reset_index(drop=True)
        )

        # print(results_df)
        return results_df

    def predict(self):
        bottom_n = self.ops["bottom_n"]
        top_n = self.ops["top_n"]
        lag = self.ops["lag"]

        def value_investing(df_bars, sentiment_df, N=1):
            for session_of_interest in range(N):
                results_df = self.identify_firm(df_bars, sentiment_df, session_of_interest=session_of_interest)
                print(results_df)

        print("Predicting train set")
        train_seen_only = self.bars_seen_train[self.bars_seen_train["bar_ix"] <= 49].copy()
        train_preds     = value_investing(self.bars_seen_train, self.)
        train_results   = backtest(train_preds.rename(columns={"target_position": "volume"})[["session", "volume"]])

        print("Predicting test set")
        # ── Apply to both test sets ───────────────────────────────────────────────
        print("Predicting public")
        public_preds  = autocorr_strategy(self.bars_seen_public_test)
        print("Predicting private")
        private_preds = autocorr_strategy(self.bars_seen_private_test)

        public_preds[["session", "target_position"]].to_csv("predictions_public_autocorr25.csv",  index=False)
        private_preds[["session", "target_position"]].to_csv("predictions_private_autocorr25.csv", index=False)
        print("Saved predictions_public_autocorr25.csv  and  predictions_private_autocorr25.csv")

        submission_filename = f"{time.time()}_predictions_autocorr25_all.csv"
        all_predictions = pd.concat([public_preds, private_preds], ignore_index=True)
        all_predictions.to_csv(submission_filename, index=False)
        print(f"Submission is available at {submission_filename}")



In [31]:
ops = {
    "lag": 25, "top_n" : 300, "bottom_n" : 200
}

model = Model(ops)

bars_seen_train: (50000, 6)
bars_unseen_train: (50000, 6)
bars_seen_public_test: (500000, 6)
bars_seen_private_test: (500000, 6)
headlines_seen_train: (9740, 3)
headlines_unseen_train: (7631, 3)
headlines_seen_public_test: (99308, 3)
headlines_seen_private_test: (99148, 3)


In [32]:
model.predict()

Predicting train set
  Sessions evaluated : 1000
  Sharpe ratio       : +2.0978
  Mean PnL           : +0.001925
  Std  PnL           : 0.014681
  Win rate           : 51.30%
Predicting test set
Predicting public
Predicting private
Saved predictions_public_autocorr25.csv  and  predictions_private_autocorr25.csv
Submission is available at 1776523498.158891_predictions_autocorr25_all.csv
